In [ ]:
import json
import os
import tkinter as tk
from tkinter import ttk, messagebox



class Member:
    def _init_(self, name, id, age, phone, trainer=None, membership=None):
        self.name = name
        self.__id = id
        self.age = age
        self.phone = phone
        self.trainer = trainer
        self.membership = membership

    def get_id(self):
        return self.__id

    def _str_(self):
        trainer_name = self.trainer.name if self.trainer else "None"
        membership_status = self.membership.status if self.membership else "None"

        return (
            f"name : {self.name} , "
            f"id : {self.__id} , "
            f"age : {self.age} , "
            f"phone : {self.phone} , "
            f"trainer : {trainer_name} , "
            f"membership : {membership_status}"
        )



class Trainer:
    def _init_(self, name, id):
        self.name = name
        self.__id = id
        self.members = []

    def get_id(self):
        return self.__id

    def _str_(self):
        return (
            f"name : {self.name} , "
            f"id : {self.__id} , "
            f"members : {len(self.members)}"
        )

    def add_member(self, member):
        if member not in self.members:
            self.members.append(member)



class Membership:
    def _init_(
        self,
        price,
        start_date,
        end_date,
        member,
        kind_of_membership,
        status="Active"
    ):
        self.__price = price
        self.start_date = start_date
        self.end_date = end_date
        self.member = member
        self.kind_of_membership = kind_of_membership
        self.status = status

    def get_price(self):
        return self.__price

    def _str_(self):
        return (
            f"price : {self.__price} , "
            f"start_date : {self.start_date} , "
            f"end_date : {self.end_date} , "
            f"member : {self.member.name} , "
            f"type : {self.kind_of_membership} , "
            f"status : {self.status}"
        )



class Payment:
    def _init_(self, member, date, amount, paid):
        self.amount = amount
        self.date = date
        self.member = member
        self.paid = paid

    def _str_(self):
        return (
            f"amount : {self.amount} , "
            f"date : {self.date} , "
            f"member : {self.member.name} , "
            f"method : {self.paid}"
        )

    def pay(self):
        if self.amount <= 0:
            raise ValueError(
                "payment amount must be greater than zero"
            )

        return True




class Attendance:
    def _init_(self, date, member):
        self.date = date
        self.member = member

    def _str_(self):
        return (
            f"date : {self.date} , "
            f"member : {self.member.name}"
        )

    def add_attendance(self):

        if self.member.membership is None:
            raise ValueError(
                "the member has no membership"
            )

        if self.member.membership.status == "Expired":
            raise ValueError(
                "the membership is expired"
            )

        return True




class Gym:

    def _init_(
        self,
        members=None,
        trainers=None,
        memberships=None,
        payments=None,
        attendance=None
    ):

        self.members = members if members is not None else []
        self.trainers = trainers if trainers is not None else []
        self.memberships = (
            memberships if memberships is not None else []
        )
        self.payments = (
            payments if payments is not None else []
        )
        self.attendance = (
            attendance if attendance is not None else []
        )



    def find_member(self, member_id):

        for member in self.members:

            if member.get_id() == member_id:
                return member

        return None



    def find_trainer(self, trainer_id):

        for trainer in self.trainers:

            if trainer.get_id() == trainer_id:
                return trainer

        return None



    def add_member(self, member):

        if self.find_member(member.get_id()) is not None:
            raise ValueError(
                "Member ID already exists"
            )

        self.members.append(member)


    def update_member(
        self,
        member_id,
        new_name,
        new_age,
        new_phone
    ):

        member = self.find_member(member_id)

        if member is None:
            raise ValueError(
                "Member not found"
            )

        member.name = new_name
        member.age = new_age
        member.phone = new_phone



    def add_trainer(self, trainer):

        if self.find_trainer(trainer.get_id()) is not None:
            raise ValueError(
                "Trainer ID already exists"
            )

        self.trainers.append(trainer)



    def add_membership(self, new_membership):

        member = new_membership.member

        if (
            member.membership is not None
            and member.membership.status == "Active"
        ):
            raise ValueError(
                "The member already has an active membership"
            )

        self.memberships.append(new_membership)

        member.membership = new_membership



    def add_payment(self, payment):

        payment.pay()

        self.payments.append(payment)


    def record_attendance(self, attendance):

        attendance.add_attendance()

        self.attendance.append(attendance)


    def assign_trainer(self, member, trainer):

        if member.trainer is not None:

            old_trainer = member.trainer

            if member in old_trainer.members:
                old_trainer.members.remove(member)

        member.trainer = trainer

        trainer.add_member(member)



    def display_members(self):

        for member in self.members:
            print(member)

    def display_trainers(self):

        for trainer in self.trainers:
            print(trainer)

    def display_memberships(self):

        for membership in self.memberships:
            print(membership)

    def display_payments(self):

        for payment in self.payments:
            print(payment)

    def display_attendance(self):

        for attendance in self.attendance:
            print(attendance)


    def save_data(self, filename="gym_data.json"):

        data = {
            "members": [],
            "trainers": [],
            "payments": [],
            "attendance": []
        }



        for member in self.members:

            member_data = {
                "id": member.get_id(),
                "name": member.name,
                "age": member.age,
                "phone": member.phone,
                "trainer_id": (
                    member.trainer.get_id()
                    if member.trainer
                    else None
                ),
                "membership": None
            }

            if member.membership:

                member_data["membership"] = {
                    "price": member.membership.get_price(),
                    "start_date": member.membership.start_date,
                    "end_date": member.membership.end_date,
                    "kind": member.membership.kind_of_membership,
                    "status": member.membership.status
                }

            data["members"].append(member_data)



        for trainer in self.trainers:

            trainer_data = {
                "id": trainer.get_id(),
                "name": trainer.name
            }

            data["trainers"].append(trainer_data)



        for payment in self.payments:

            payment_data = {
                "member_id": payment.member.get_id(),
                "date": payment.date,
                "amount": payment.amount,
                "paid": payment.paid
            }

            data["payments"].append(payment_data)


        for attendance in self.attendance:

            attendance_data = {
                "member_id": attendance.member.get_id(),
                "date": attendance.date
            }

            data["attendance"].append(attendance_data)



        with open(
            filename,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                data,
                file,
                indent=4,
                ensure_ascii=False
            )


    def load_data(self, filename="gym_data.json"):

        if not os.path.exists(filename):
            return

        with open(
            filename,
            "r",
            encoding="utf-8"
        ) as file:

            data = json.load(file)

        self.members = []
        self.trainers = []
        self.memberships = []
        self.payments = []
        self.attendance = []



        for member_data in data.get("members", []):

            member = Member(
                member_data["name"],
                member_data["id"],
                member_data["age"],
                member_data["phone"]
            )

            self.members.append(member)



        for trainer_data in data.get("trainers", []):

            trainer = Trainer(
                trainer_data["name"],
                trainer_data["id"]
            )

            self.trainers.append(trainer)


        for member_data in data.get("members", []):

            member = self.find_member(
                member_data["id"]
            )

            trainer_id = member_data.get(
                "trainer_id"
            )

            if trainer_id is not None:

                trainer = self.find_trainer(
                    trainer_id
                )

                if trainer:

                    self.assign_trainer(
                        member,
                        trainer
                    )



        for member_data in data.get("members", []):

            membership_data = member_data.get(
                "membership"
            )

            if membership_data is not None:

                member = self.find_member(
                    member_data["id"]
                )

                membership = Membership(
                    membership_data["price"],
                    membership_data["start_date"],
                    membership_data["end_date"],
                    member,
                    membership_data["kind"],
                    membership_data["status"]
                )

                member.membership = membership

                self.memberships.append(
                    membership
                )



        for payment_data in data.get(
            "payments",
            []
        ):

            member = self.find_member(
                payment_data["member_id"]
            )

            if member:

                payment = Payment(
                    member,
                    payment_data["date"],
                    payment_data["amount"],
                    payment_data["paid"]
                )

                self.payments.append(
                    payment
                )



        for attendance_data in data.get(
            "attendance",
            []
        ):

            member = self.find_member(
                attendance_data["member_id"]
            )

            if member:

                attendance = Attendance(
                    attendance_data["date"],
                    member
                )

                self.attendance.append(
                    attendance
                )




gym = Gym()

gym.load_data()





def menu():

    while True:

        print("\n")
        print("========== GYM MANAGEMENT SYSTEM ==========")
        print("1. Add Member")
        print("2. Update Member")
        print("3. Add Trainer")
        print("4. Create Membership")
        print("5. Add Payment")
        print("6. Record Attendance")
        print("7. Assign Trainer")
        print("8. Display Members")
        print("9. Display Trainers")
        print("10. Display Memberships")
        print("11. Display Payments")
        print("12. Display Attendance")
        print("13. Save Data")
        print("0. Exit")
        print("===========================================")

        choice = input("Enter your choice: ")

        try:


            if choice == "1":

                name = input("Name: ")
                member_id = input("ID: ")
                age = int(input("Age: "))
                phone = input("Phone: ")

                member = Member(
                    name,
                    member_id,
                    age,
                    phone
                )

                gym.add_member(member)

                gym.save_data()

                print("Member added successfully")



            elif choice == "2":

                member_id = input("Enter Member ID: ")
                name = input("New Name: ")
                age = int(input("New Age: "))
                phone = input("New Phone: ")

                gym.update_member(
                    member_id,
                    name,
                    age,
                    phone
                )

                gym.save_data()

                print("Member updated successfully")



            elif choice == "3":

                name = input("Trainer Name: ")
                trainer_id = input("Trainer ID: ")

                trainer = Trainer(
                    name,
                    trainer_id
                )

                gym.add_trainer(trainer)

                gym.save_data()

                print("Trainer added successfully")



            elif choice == "4":

                member_id = input(
                    "Enter Member ID: "
                )

                member = gym.find_member(
                    member_id
                )

                if member is None:
                    raise ValueError(
                        "Member not found"
                    )

                price = float(
                    input("Price: ")
                )

                start_date = input(
                    "Start Date: "
                )

                end_date = input(
                    "End Date: "
                )

                kind = input(
                    "Membership Type: "
                )

                membership = Membership(
                    price,
                    start_date,
                    end_date,
                    member,
                    kind
                )

                gym.add_membership(
                    membership
                )

                gym.save_data()

                print(
                    "Membership created successfully"
                )



            elif choice == "5":

                member_id = input(
                    "Enter Member ID: "
                )

                member = gym.find_member(
                    member_id
                )

                if member is None:
                    raise ValueError(
                        "Member not found"
                    )

                date = input(
                    "Date: "
                )

                amount = float(
                    input("Amount: ")
                )

                paid = input(
                    "Payment Method: "
                )

                payment = Payment(
                    member,
                    date,
                    amount,
                    paid
                )

                gym.add_payment(
                    payment
                )

                gym.save_data()

                print(
                    "Payment added successfully"
                )



            elif choice == "6":

                member_id = input(
                    "Enter Member ID: "
                )

                member = gym.find_member(
                    member_id
                )

                if member is None:
                    raise ValueError(
                        "Member not found"
                    )

                date = input(
                    "Date: "
                )

                attendance = Attendance(
                    date,
                    member
                )

                gym.record_attendance(
                    attendance
                )

                gym.save_data()

                print(
                    "Attendance recorded successfully"
                )



            elif choice == "7":

                member_id = input(
                    "Member ID: "
                )

                trainer_id = input(
                    "Trainer ID: "
                )

                member = gym.find_member(
                    member_id
                )

                trainer = gym.find_trainer(
                    trainer_id
                )

                if member is None:
                    raise ValueError(
                        "Member not found"
                    )

                if trainer is None:
                    raise ValueError(
                        "Trainer not found"
                    )

                gym.assign_trainer(
                    member,
                    trainer
                )

                gym.save_data()

                print(
                    "Trainer assigned successfully"
                )


            elif choice == "8":

                gym.display_members()

            elif choice == "9":

                gym.display_trainers()

            elif choice == "10":

                gym.display_memberships()

            elif choice == "11":

                gym.display_payments()

            elif choice == "12":

                gym.display_attendance()

            elif choice == "13":

                gym.save_data()

                print("Data saved successfully")

            elif choice == "0":

                gym.save_data()

                print("Goodbye!")
                break

            else:

                print("Invalid choice")

        except ValueError as error:

            print("Error:", error)




class GymGUI:

    def _init_(self, root):

        self.root = root

        self.root.title(
            "Gym Management System"
        )

        self.root.geometry(
            "850x600"
        )

        self.notebook = ttk.Notebook(
            root
        )

        self.notebook.pack(
            fill="both",
            expand=True
        )

        # Create tabs

        self.create_members_tab()
        self.create_trainers_tab()
        self.create_membership_tab()
        self.create_payment_tab()
        self.create_attendance_tab()
        self.create_assign_tab()
        self.create_display_tab()



    def create_members_tab(self):

        frame = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            frame,
            text="Members"
        )

        ttk.Label(
            frame,
            text="Name"
        ).pack()

        self.member_name = ttk.Entry(
            frame
        )

        self.member_name.pack()

        ttk.Label(
            frame,
            text="ID"
        ).pack()

        self.member_id = ttk.Entry(
            frame
        )

        self.member_id.pack()

        ttk.Label(
            frame,
            text="Age"
        ).pack()

        self.member_age = ttk.Entry(
            frame
        )

        self.member_age.pack()

        ttk.Label(
            frame,
            text="Phone"
        ).pack()

        self.member_phone = ttk.Entry(
            frame
        )

        self.member_phone.pack()

        ttk.Button(
            frame,
            text="Add Member",
            command=self.add_member
        ).pack(pady=10)

        ttk.Button(
            frame,
            text="Update Member",
            command=self.update_member
        ).pack()


    def add_member(self):

        try:

            member = Member(
                self.member_name.get(),
                self.member_id.get(),
                int(self.member_age.get()),
                self.member_phone.get()
            )

            gym.add_member(member)

            gym.save_data()

            messagebox.showinfo(
                "Success",
                "Member added successfully"
            )

            self.clear_member_fields()

            self.refresh_display()

        except ValueError as error:

            messagebox.showerror(
                "Error",
                str(error)
            )



    def update_member(self):

        try:

            gym.update_member(
                self.member_id.get(),
                self.member_name.get(),
                int(self.member_age.get()),
                self.member_phone.get()
            )

            gym.save_data()

            messagebox.showinfo(
                "Success",
                "Member updated successfully"
            )

            self.refresh_display()

        except ValueError as error:

            messagebox.showerror(
                "Error",
                str(error)
            )


    def create_trainers_tab(self):

        frame = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            frame,
            text="Trainers"
        )

        ttk.Label(
            frame,
            text="Trainer Name"
        ).pack()

        self.trainer_name = ttk.Entry(
            frame
        )

        self.trainer_name.pack()

        ttk.Label(
            frame,
            text="Trainer ID"
        ).pack()

        self.trainer_id = ttk.Entry(
            frame
        )

        self.trainer_id.pack()

        ttk.Button(
            frame,
            text="Add Trainer",
            command=self.add_trainer
        ).pack(pady=10)


    def add_trainer(self):

        try:

            trainer = Trainer(
                self.trainer_name.get(),
                self.trainer_id.get()
            )

            gym.add_trainer(
                trainer
            )

            gym.save_data()

            messagebox.showinfo(
                "Success",
                "Trainer added successfully"
            )

            self.refresh_display()

        except ValueError as error:

            messagebox.showerror(
                "Error",
                str(error)
            )


    def create_membership_tab(self):

        frame = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            frame,
            text="Membership"
        )

        ttk.Label(
            frame,
            text="Member ID"
        ).pack()

        self.mem_member_id = ttk.Entry(
            frame
        )

        self.mem_member_id.pack()

        ttk.Label(
            frame,
            text="Price"
        ).pack()

        self.mem_price = ttk.Entry(
            frame
        )

        self.mem_price.pack()

        ttk.Label(
            frame,
            text="Start Date"
        ).pack()

        self.mem_start = ttk.Entry(
            frame
        )

        self.mem_start.pack()

        ttk.Label(
            frame,
            text="End Date"
        ).pack()

        self.mem_end = ttk.Entry(
            frame
        )

        self.mem_end.pack()

        ttk.Label(
            frame,
            text="Type"
        ).pack()

        self.mem_type = ttk.Entry(
            frame
        )

        self.mem_type.pack()

        ttk.Button(
            frame,
            text="Create Membership",
            command=self.add_membership
        ).pack(pady=10)



    def add_membership(self):

        try:

            member = gym.find_member(
                self.mem_member_id.get()
            )

            if member is None:
                raise ValueError(
                    "Member not found"
                )

            membership = Membership(
                float(self.mem_price.get()),
                self.mem_start.get(),
                self.mem_end.get(),
                member,
                self.mem_type.get()
            )

            gym.add_membership(
                membership
            )

            gym.save_data()

            messagebox.showinfo(
                "Success",
                "Membership created successfully"
            )

            self.refresh_display()

        except ValueError as error:

            messagebox.showerror(
                "Error",
                str(error)
            )

    def create_payment_tab(self):

        frame = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            frame,
            text="Payment"
        )

        ttk.Label(
            frame,
            text="Member ID"
        ).pack()

        self.payment_member = ttk.Entry(
            frame
        )

        self.payment_member.pack()

        ttk.Label(
            frame,
            text="Date"
        ).pack()

        self.payment_date = ttk.Entry(
            frame
        )

        self.payment_date.pack()

        ttk.Label(
            frame,
            text="Amount"
        ).pack()

        self.payment_amount = ttk.Entry(
            frame
        )

        self.payment_amount.pack()

        ttk.Label(
            frame,
            text="Payment Method"
        ).pack()

        self.payment_method = ttk.Entry(
            frame
        )

        self.payment_method.pack()

        ttk.Button(
            frame,
            text="Add Payment",
            command=self.add_payment
        ).pack(pady=10)



    def add_payment(self):

        try:

            member = gym.find_member(
                self.payment_member.get()
            )

            if member is None:
                raise ValueError(
                    "Member not found"
                )

            payment = Payment(
                member,
                self.payment_date.get(),
                float(self.payment_amount.get()),
                self.payment_method.get()
            )

            gym.add_payment(
                payment
            )

            gym.save_data()

            messagebox.showinfo(
                "Success",
                "Payment added successfully"
            )

            self.refresh_display()

        except ValueError as error:

            messagebox.showerror(
                "Error",
                str(error)
            )


    def create_attendance_tab(self):

        frame = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            frame,
            text="Attendance"
        )

        ttk.Label(
            frame,
            text="Member ID"
        ).pack()

        self.att_member = ttk.Entry(
            frame
        )

        self.att_member.pack()

        ttk.Label(
            frame,
            text="Date"
        ).pack()

        self.att_date = ttk.Entry(
            frame
        )

        self.att_date.pack()

        ttk.Button(
            frame,
            text="Record Attendance",
            command=self.record_attendance
        ).pack(pady=10)



    def record_attendance(self):

        try:

            member = gym.find_member(
                self.att_member.get()
            )

            if member is None:
                raise ValueError(
                    "Member not found"
                )

            attendance = Attendance(
                self.att_date.get(),
                member
            )

            gym.record_attendance(
                attendance
            )

            gym.save_data()

            messagebox.showinfo(
                "Success",
                "Attendance recorded successfully"
            )

            self.refresh_display()

        except ValueError as error:

            messagebox.showerror(
                "Error",
                str(error)
            )



    def create_assign_tab(self):

        frame = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            frame,
            text="Assign Trainer"
        )

        ttk.Label(
            frame,
            text="Member ID"
        ).pack()

        self.assign_member = ttk.Entry(
            frame
        )

        self.assign_member.pack()

        ttk.Label(
            frame,
            text="Trainer ID"
        ).pack()

        self.assign_trainer_id = ttk.Entry(
            frame
        )

        self.assign_trainer_id.pack()

        ttk.Button(
            frame,
            text="Assign Trainer",
            command=self.assign_trainer
        ).pack(pady=10)


    def assign_trainer(self):

        try:

            member = gym.find_member(
                self.assign_member.get()
            )

            trainer = gym.find_trainer(
                self.assign_trainer_id.get()
            )

            if member is None:
                raise ValueError(
                    "Member not found"
                )

            if trainer is None:
                raise ValueError(
                    "Trainer not found"
                )

            gym.assign_trainer(
                member,
                trainer
            )

            gym.save_data()

            messagebox.showinfo(
                "Success",
                "Trainer assigned successfully"
            )

            self.refresh_display()

        except ValueError as error:

            messagebox.showerror(
                "Error",
                str(error)
            )


    def create_display_tab(self):

        frame = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            frame,
            text="Display Data"
        )

        self.display_text = tk.Text(
            frame,
            width=100,
            height=30
        )

        self.display_text.pack(
            fill="both",
            expand=True
        )

        ttk.Button(
            frame,
            text="Refresh",
            command=self.refresh_display
        ).pack(pady=5)

        self.refresh_display()



    def refresh_display(self):

        if not hasattr(
            self,
            "display_text"
        ):
            return

        self.display_text.delete(
            "1.0",
            tk.END
        )

        self.display_text.insert(
            tk.END,
            "========== MEMBERS ==========\n\n"
        )

        for member in gym.members:

            self.display_text.insert(
                tk.END,
                str(member) + "\n"
            )

        self.display_text.insert(
            tk.END,
            "\n========== TRAINERS ==========\n\n"
        )

        for trainer in gym.trainers:

            self.display_text.insert(
                tk.END,
                str(trainer) + "\n"
            )

        self.display_text.insert(
            tk.END,
            "\n========== MEMBERSHIPS ==========\n\n"
        )

        for membership in gym.memberships:

            self.display_text.insert(
                tk.END,
                str(membership) + "\n"
            )

        self.display_text.insert(
            tk.END,
            "\n========== PAYMENTS ==========\n\n"
        )

        for payment in gym.payments:

            self.display_text.insert(
                tk.END,
                str(payment) + "\n"
            )

        self.display_text.insert(
            tk.END,
            "\n========== ATTENDANCE ==========\n\n"
        )

        for attendance in gym.attendance:

            self.display_text.insert(
                tk.END,
                str(attendance) + "\n"
            )


    def clear_member_fields(self):

        self.member_name.delete(
            0,
            tk.END
        )

        self.member_id.delete(
            0,
            tk.END
        )

        self.member_age.delete(
            0,
            tk.END
        )

        self.member_phone.delete(
            0,
            tk.END
        )




root = tk.Tk()

app = GymGUI(root)


def close_program():

    gym.save_data()

    root.destroy()


root.protocol(
    "WM_DELETE_WINDOW",
    close_program
)

root.mainloop()